<a href="https://colab.research.google.com/github/liorg/michal/blob/main/Scenariobot_Poc_regex_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 76.8 MB/s eta 0:00:00


In [1]:
!pip -q install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 144.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.8 MB/s eta 0:00:00


In [2]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen3-4B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=quantization_config
)

print("allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2),
      "GB")

print("reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2),
      "GB")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

allocated: 2.66 GB
reserved: 2.71 GB


In [3]:

RULES = r"""
You are a multilingual text-condition parser.

Your task is NOT to generate regular expressions.

Convert the user's natural-language requirement into structured JSON
that describes the required text conditions.

You understand:
- Hebrew
- English
- Arabic
- Russian


==================================================
1. SUPPORTED CONDITION TYPES
==================================================

You may ONLY use these condition types:

- starts_with
- ends_with
- contains
- contains_after
- exact
- digits
- letters
- one_of
- not_contains
- space
- literal

Do not invent additional condition types.


==================================================
2. GENERAL RULES
==================================================

1. Preserve literal values exactly as written by the user.

2. Preserve the ORDER of conditions.

3. Do NOT generate regex.

4. Do NOT output regex syntax such as:
   ^
   $
   \d
   .*
   \b

5. Do not invent conditions that were not requested.

6. Do not omit explicitly described components.

7. Separators such as spaces, hyphens, colons and slashes are
   separate conditions when they are part of a structured sequence.

8. Every condition MUST contain exactly these fields:

   {
     "type": "...",
     "value": null,
     "count": null
   }

9. Use null when value or count is not relevant.


==================================================
3. STARTS_WITH
==================================================

When the user says the text starts with a value:

{
  "type": "starts_with",
  "value": "VALUE",
  "count": null
}

Examples of meaning:

Hebrew:
- מתחיל ב
- מתחילה ב
- בתחילת הטקסט
- בתחילת ההודעה

English:
- starts with
- begins with

Russian:
- начинается с

Arabic:
- يبدأ بـ

IMPORTANT:

starts_with by itself does NOT restrict the entire input.

Example:

User:
הטקסט חייב להתחיל ב"שלום"

Result:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "שלום",
      "count": null
    }
  ],
  "whole_input": false
}


==================================================
4. ENDS_WITH
==================================================

When the user says the text must end with a value:

{
  "type": "ends_with",
  "value": "VALUE",
  "count": null
}

Examples of meaning:

Hebrew:
- מסתיים ב
- בסוף ההודעה
- בסוף הטקסט

English:
- ends with

Russian:
- заканчивается на

Arabic:
- ينتهي بـ

ends_with by itself does NOT restrict the entire input.

Therefore normally:

"whole_input": false


==================================================
5. CONTAINS
==================================================

Use contains when a value may occur anywhere in the input and
NO ordering relationship is specified.

{
  "type": "contains",
  "value": "VALUE",
  "count": null
}

Examples:

Hebrew:
- מכיל
- חייב להופיע
- איפשהו

English:
- contains
- appears somewhere

Russian:
- содержит

Arabic:
- يحتوي

contains normally means:

"whole_input": false


==================================================
6. CONTAINS_AFTER — ORDER IS CRITICAL
==================================================

contains_after is DIFFERENT from contains.

Use contains_after whenever the user explicitly says that one value
must occur AFTER a previously described condition.

ORDER WORDS MUST trigger contains_after.

Hebrew:
- בהמשך
- אחר כך
- לאחר מכן
- מאוחר יותר

English:
- later
- afterwards
- after that

Russian:
- позже
- затем
- после этого

Arabic:
- لاحقًا
- بعد ذلك

IMPORTANT:

When these words describe a later occurrence,
DO NOT use contains.
Use contains_after.

Example:

User:
הטקסט מתחיל ב"שלום" ובהמשך מופיע "עולם"

Correct:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "שלום",
      "count": null
    },
    {
      "type": "contains_after",
      "value": "עולם",
      "count": null
    }
  ],
  "whole_input": false
}

WRONG:

{
  "type": "contains",
  "value": "עולם"
}


Another example:

User:
הטקסט מכיל "START" ובהמשך צריך להופיע "STOP"

Correct order:

contains("START")
contains_after("STOP")


IMPORTANT DISTINCTION:

Words such as:
- אחריו
- אחריה
- then
- followed by

may describe an IMMEDIATE next component in a structured format.

For example:

"קוד", אחריו רווח ואז 5 ספרות

is a structured sequence:

starts_with("קוד")
space
digits(5)

Do NOT convert the space or digits into contains_after.


==================================================
7. EXACT
==================================================

Use exact ONLY when the ENTIRE response must equal one literal value.

{
  "type": "exact",
  "value": "VALUE",
  "count": null
}

Typical wording:

Hebrew:
- בדיוק
- רק
- בלבד
- היחידה שמותר
- התשובה היחידה

English:
- exactly
- only

Russian:
- только
- точно

Arabic:
- فقط
- بالضبط

Example:

User:
התשובה היחידה שמותר לקבל היא "בוצע"

Correct:

{
  "conditions": [
    {
      "type": "exact",
      "value": "בוצע",
      "count": null
    }
  ],
  "whole_input": true
}


IMPORTANT:

NEVER use exact as a component inside a sequence.

Example:

User:
ההודעה מתחילה ב"לקוח", אחריה רווח ואז "פעיל"

Correct:

starts_with("לקוח")
space
literal("פעיל")

WRONG:

starts_with("לקוח")
space
exact("פעיל")


==================================================
8. ONE_OF / ALTERNATIVES
==================================================

When multiple literal values are alternatives,
represent them as ONE one_of condition.

Alternative words include:

Hebrew:
- או

English:
- or

Russian:
- или

Arabic:
- أو

Example:

User:
התגובה חייבת להיות בדיוק "כן" או "לא"

Correct:

{
  "conditions": [
    {
      "type": "one_of",
      "value": ["כן", "לא"],
      "count": null
    }
  ],
  "whole_input": true
}

NEVER create multiple exact conditions for alternatives.


Example:

User:
קבל רק "אישור", "ביטול" או "המתנה"

Correct:

{
  "conditions": [
    {
      "type": "one_of",
      "value": ["אישור", "ביטול", "המתנה"],
      "count": null
    }
  ],
  "whole_input": true
}


==================================================
9. DIGITS
==================================================

Use digits when the requirement refers to numeric digits.

{
  "type": "digits",
  "value": null,
  "count": N
}

CRITICAL LANGUAGE MAPPING:

Hebrew:
- ספרה
- ספרות
= digits

English:
- digit
- digits
= digits

Russian:
- цифра
- цифры
= digits

Arabic:
- رقم
- أرقام
= digits


CRITICAL:

Hebrew "ספרות" ALWAYS means digits in this context.

NEVER interpret "ספרות" as letters.


Example:

User:
התגובה חייבת להיות בדיוק 4 ספרות

Correct:

{
  "conditions": [
    {
      "type": "digits",
      "value": null,
      "count": 4
    }
  ],
  "whole_input": true
}


==================================================
10. LETTERS
==================================================

Use letters when the requirement explicitly refers to letters.

{
  "type": "letters",
  "value": null,
  "count": N
}

Language mapping:

Hebrew:
- אות
- אותיות

English:
- letter
- letters

Russian:
- буква
- буквы

Arabic:
- حرف
- حروف

Example:

User:
התגובה חייבת להכיל בדיוק 3 אותיות בלבד

Correct:

{
  "conditions": [
    {
      "type": "letters",
      "value": null,
      "count": 3
    }
  ],
  "whole_input": true
}


==================================================
11. SPACE
==================================================

If the user explicitly describes a space between components,
create a separate space condition.

{
  "type": "space",
  "value": null,
  "count": null
}

Language examples:

Hebrew:
- רווח
- אחריו רווח
- אחריה רווח

English:
- space
- followed by a space

Russian:
- пробел

Arabic:
- مسافة


Example:

User:
התגובה מתחילה ב"קוד", אחר כך רווח ואז בדיוק 5 ספרות

Correct:

starts_with("קוד")
space
digits(5)

The space MUST NOT be omitted.


==================================================
12. LITERAL
==================================================

Use literal for a fixed value or separator that occurs as a component
inside a structured sequence.

{
  "type": "literal",
  "value": "VALUE",
  "count": null
}

Example:

User:
ההודעה מתחילה ב"לקוח", אחריה רווח ואז "פעיל"

Correct:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "לקוח",
      "count": null
    },
    {
      "type": "space",
      "value": null,
      "count": null
    },
    {
      "type": "literal",
      "value": "פעיל",
      "count": null
    }
  ],
  "whole_input": true
}


==================================================
13. SYMBOL MAPPING
==================================================

When the user names a symbol, output the ACTUAL ASCII symbol.

Use these mappings exactly.


Hebrew:

מקף
=> "-"

נקודתיים
=> ":"

לוכסן
=> "/"


English:

hyphen
=> "-"

dash
=> "-"

colon
=> ":"

slash
=> "/"


Russian:

дефис
=> "-"

тире
=> "-"

двоеточие
=> ":"

слэш
=> "/"


Arabic:

شرطة
=> "-"

نقطتان
=> ":"

شرطة مائلة
=> "/"


CRITICAL:

Never substitute another character.

For example:

נקודתיים MUST be ":"
NEVER "."

לוכסן MUST be "/"
NEVER space
NEVER another Unicode character.


==================================================
14. NOT_CONTAINS
==================================================

When the user says a value must NOT appear:

{
  "type": "not_contains",
  "value": "VALUE",
  "count": null
}

Examples:

Hebrew:
- אסור להכיל
- אסור שיופיע
- אסור שתכיל

English:
- must not contain
- cannot contain

Russian:
- не должен содержать

Arabic:
- يجب ألا يحتوي

not_contains by itself does NOT restrict the entire input.

Therefore normally:

"whole_input": false


==================================================
15. WHOLE_INPUT
==================================================

whole_input tells whether the COMPLETE input must conform to the
complete described format.


Set:

"whole_input": true

when the user describes a CLOSED complete response or format.


Examples:


A. Exact value

"בדיוק כן"

=> true


B. Closed alternatives

"רק כן או לא"

=> true


C. Exact number of digits only

"רק 6 ספרות"

=> true


D. Exact number of letters only

"בדיוק 3 אותיות בלבד"

=> true


E. Complete structured format

"קוד", space, exactly 5 digits

=> true


F. Complete structured format

"ORDER", hyphen, exactly 6 digits

=> true


G. Complete structured format

"ABC", slash, exactly 3 digits

=> true


IMPORTANT:

When the user describes a contiguous structured format beginning
with a fixed prefix and continuing through a final fixed-length
component, whole_input MUST be true.

Example:

User:
הטקסט מתחיל ב"ABC", אחר כך לוכסן ואז 3 ספרות

Correct:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "ABC",
      "count": null
    },
    {
      "type": "literal",
      "value": "/",
      "count": null
    },
    {
      "type": "digits",
      "value": null,
      "count": 3
    }
  ],
  "whole_input": true
}


Set:

"whole_input": false

when the requirement only constrains part of otherwise unrestricted text.


Examples:

starts with "שלום"
=> false

ends with "תודה"
=> false

contains "ABC"
=> false

does not contain "ERROR"
=> false


Ordered content with unspecified text between components is also false.

Example:

starts with "BEGIN"
later contains "DATA"
ends with "END"

=> whole_input = false


==================================================
16. OUTPUT STRUCTURE — CRITICAL
==================================================

Return ONLY valid JSON.

The response MUST ALWAYS be a JSON OBJECT.

The first character MUST be:

{

The last character MUST be:

}


The exact top-level structure is:

{
  "conditions": [
    {
      "type": "condition_type",
      "value": null,
      "count": null
    }
  ],
  "whole_input": false
}


CRITICAL:

NEVER return the conditions array directly.

WRONG:

[
  {
    "type": "starts_with",
    "value": "A",
    "count": null
  }
]


CORRECT:

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "A",
      "count": null
    }
  ],
  "whole_input": false
}


"whole_input" MUST ALWAYS exist.

"whole_input" MUST ALWAYS be a top-level property.

NEVER put whole_input inside the conditions array.


==================================================
17. FINAL VALIDATION BEFORE RESPONDING
==================================================

Before returning the JSON, silently verify:

1. Is the output a JSON object?
2. Does it contain "conditions"?
3. Does it contain "whole_input"?
4. Is every condition type supported?
5. Does every condition contain type, value and count?
6. Did I preserve the order?
7. Did I preserve all literal values?
8. Did I preserve explicitly requested spaces and separators?
9. Did I use contains_after when the user explicitly described a later occurrence?
10. Did I distinguish digits from letters correctly?
11. Did I use exact only for an entire exact response?
12. Did I use literal instead of exact inside structured sequences?
13. Is whole_input correct?
14. Is the JSON syntactically valid?

Return ONLY the JSON.
Do not explain the result.
Do not use Markdown.
Do not use code fences.
"""

In [13]:
print("RULES tokens:", len(tokenizer.encode(RULES)))

RULES tokens: 3158


In [5]:

SEPARATORS = {
    "hyphen": "-",
    "colon": ":",
    "slash": "/"
}


def escape_regex(value):
    if value is None:
        return ""

    return re.sub(
        r'([.*+?^${}()|[\]\\])',
        r'\\\1',
        value
    )


def compile_condition(data):
    conditions = data.get("conditions", [])
    whole_input = data.get("whole_input", False)

    parts = []

    for condition in conditions:
        condition_type = condition.get("type")
        value = condition.get("value")
        count = condition.get("count")

        if condition_type == "starts_with":
            parts.append(
                "^" + escape_regex(value)
            )

        elif condition_type == "ends_with":
            parts.append(
                escape_regex(value) + "$"
            )

        elif condition_type == "contains":
            parts.append(
                ".*" + escape_regex(value)
            )

        elif condition_type == "contains_after":
            parts.append(
                ".*" + escape_regex(value)
            )

        elif condition_type == "exact":
            parts.append(
                "^" + escape_regex(value) + "$"
            )

        elif condition_type == "digits":
            if not isinstance(count, int) or count < 1:
                raise ValueError(
                    f"Invalid digits count: {count}"
                )

            parts.append(
                rf"\d{{{count}}}"
            )

        elif condition_type == "letters":
            if not isinstance(count, int) or count < 1:
                raise ValueError(
                    f"Invalid letters count: {count}"
                )

            parts.append(
                rf"[A-Za-z]{{{count}}}"
            )

        elif condition_type == "one_of":
            if not isinstance(value, list) or not value:
                raise ValueError(
                    "one_of requires values"
                )

            escaped = [
                escape_regex(v)
                for v in value
            ]

            parts.append(
                "(" + "|".join(escaped) + ")"
            )

        elif condition_type == "space":
            parts.append(" ")

        elif condition_type == "separator":
            symbol = SEPARATORS.get(value)

            if symbol is None:
                raise ValueError(
                    f"Unsupported separator: {value}"
                )

            parts.append(
                escape_regex(symbol)
            )

        elif condition_type == "literal":
            parts.append(
                escape_regex(value)
            )

        elif condition_type == "not_contains":
            parts.insert(
                0,
                "(?!.*" + escape_regex(value) + ")"
            )

        else:
            raise ValueError(
                f"Unsupported condition type: {condition_type}"
            )

    pattern = "".join(parts)

    if whole_input:
        if not pattern.startswith("^"):
            pattern = "^" + pattern

        if not pattern.endswith("$"):
            pattern += "$"

    return pattern

In [6]:

import json
import torch
import re

def understand_condition(request):
    messages = [
        {
            "role": "system",
            "content": """
You are a multilingual text-condition parser.

Convert natural-language requirements into structured JSON conditions.

You understand:
- Hebrew
- English
- Arabic
- Russian

Follow the CONDITION PARSING RULES exactly.
Return ONLY valid JSON.
"""
        },

        # ה-RULES שהגדרנו במקטע הקודם
        {
            "role": "system",
            "content": RULES
        },

        # הבקשה של המשתמש
        {
            "role": "user",
            "content": request
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    generated = output[0][inputs.input_ids.shape[1]:]

    raw = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    try:
        return json.loads(raw)

    except json.JSONDecodeError:
        return {
            "error": "Model returned invalid JSON",
            "raw": raw
        }

In [7]:

import json


def normalize(data):
    return json.dumps(
        data,
        ensure_ascii=False,
        sort_keys=True
    )

In [12]:

def normalize_dsl(data, request):
    if not isinstance(data, dict):
        return data

    conditions = data.get("conditions")

    if not isinstance(conditions, list):
        return data

    # 1. one_of עם אפשרות אחת = exact
    for condition in conditions:
        if (
            condition.get("type") == "one_of"
            and isinstance(condition.get("value"), list)
            and len(condition["value"]) == 1
        ):
            condition["type"] = "exact"
            condition["value"] = condition["value"][0]
            data["whole_input"] = True

    # 2. תיקון סימנים לפי מה שהמשתמש ביקש
    symbol_map = {
        # Hebrew
        "נקודתיים": ":",
        "מקף": "-",
        "לוכסן": "/",

        # English
        "colon": ":",
        "hyphen": "-",
        "slash": "/",

        # Russian
        "двоеточие": ":",
        "дефис": "-",
        "слэш": "/",

        # Arabic
        "نقطتان": ":",
        "شرطة مائلة": "/",
    }

    requested_symbol = None

    request_lower = request.lower()

    for word, symbol in symbol_map.items():
        if word.lower() in request_lower:
            requested_symbol = symbol
            break

    if requested_symbol is not None:
        for condition in conditions:
            # אם המודל המציא סימן במקום הסימן שהתבקש
            if (
                condition.get("type") == "literal"
                and isinstance(condition.get("value"), str)
                and condition["value"] in {
                    ".", ":", "-", "/", "}", "{", "–", "—"
                }
            ):
                condition["value"] = requested_symbol

            # תמיכה גם אם המודל במקרה החזיר separator
            elif condition.get("type") == "separator":
                condition["type"] = "literal"
                condition["value"] = requested_symbol

    return data

In [14]:

x = understand_condition(
    'התגובה חייבת להתחיל במילה "קוד", אחריה רווח ואז בדיוק 5 ספרות'
)

print(repr(x))

{'conditions': [{'type': 'starts_with', 'value': 'קוד', 'count': None}, {'type': 'space', 'value': None, 'count': None}, {'type': 'digits', 'value': None, 'count': 5}], 'whole_input': True}


In [ ]:

print(json.dumps(
    understand_condition(
        'התגובה חייבת להיות בדיוק "כן" או "לא"'
    ),
    ensure_ascii=False,
    indent=2
))

{
  "conditions": [
    {
      "type": "one_of",
      "value": [
        "כן",
        "לא"
      ],
      "count": null
    }
  ],
  "whole_input": true
}


In [ ]:
print(model.config.name_or_path)

Qwen/Qwen3-4B


In [ ]:

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [15]:

TEST_CASES = [

    # ============================================================
    # HEBREW — 50 hard-coded tests
    # ============================================================

    {
        "id": 1,
        "lang": "he",
        "request": 'הטקסט חייב להתחיל ב"שלום"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "שלום", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 2,
        "lang": "he",
        "request": 'ההודעה צריכה להתחיל במילה "הזמנה"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "הזמנה", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 3,
        "lang": "he",
        "request": 'הטקסט צריך להסתיים ב"תודה"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "תודה", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 4,
        "lang": "he",
        "request": 'בסוף ההודעה חייב להופיע "DONE"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "DONE", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 5,
        "lang": "he",
        "request": 'הטקסט חייב להכיל "אישור"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "אישור", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 6,
        "lang": "he",
        "request": 'איפשהו בהודעה צריכה להופיע המילה "הצלחה"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "הצלחה", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 7,
        "lang": "he",
        "request": 'התגובה חייבת להיות בדיוק "כן"',
        "expected": {
            "conditions": [
                {"type": "exact", "value": "כן", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 8,
        "lang": "he",
        "request": 'רק המילה "מאושר" מותרת בתגובה',
        "expected": {
            "conditions": [
                {"type": "exact", "value": "מאושר", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 9,
        "lang": "he",
        "request": 'התגובה יכולה להיות רק "כן" או "לא"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["כן", "לא"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 10,
        "lang": "he",
        "request": 'קבל רק "אישור", "ביטול" או "המתנה"',
        "expected": {
            "conditions": [
                {
                    "type": "one_of",
                    "value": ["אישור", "ביטול", "המתנה"],
                    "count": None
                }
            ],
            "whole_input": True
        }
    },

    {
        "id": 11,
        "lang": "he",
        "request": 'התגובה חייבת להיות בדיוק 4 ספרות',
        "expected": {
            "conditions": [
                {"type": "digits", "value": None, "count": 4}
            ],
            "whole_input": True
        }
    },

    {
        "id": 12,
        "lang": "he",
        "request": 'אני רוצה לקבל רק מספר שמורכב בדיוק מ-8 ספרות',
        "expected": {
            "conditions": [
                {"type": "digits", "value": None, "count": 8}
            ],
            "whole_input": True
        }
    },

    {
        "id": 13,
        "lang": "he",
        "request": 'התגובה חייבת להכיל בדיוק 3 אותיות בלבד',
        "expected": {
            "conditions": [
                {"type": "letters", "value": None, "count": 3}
            ],
            "whole_input": True
        }
    },

    {
        "id": 14,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"שלום" ובהמשך מופיע "עולם"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "שלום", "count": None},
                {"type": "contains_after", "value": "עולם", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 15,
        "lang": "he",
        "request": 'קודם ההודעה מתחילה ב"START" ובהמשך צריכה להופיע "END"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "START", "count": None},
                {"type": "contains_after", "value": "END", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 16,
        "lang": "he",
        "request": 'התגובה מתחילה ב"קוד", אחר כך רווח ואז בדיוק 5 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "קוד", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "digits", "value": None, "count": 5}
            ],
            "whole_input": True
        }
    },

    {
        "id": 17,
        "lang": "he",
        "request": 'הטקסט חייב להתחיל ב"ORDER", אחריו מקף ואז בדיוק 6 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "ORDER", "count": None},
                {"type": "literal", "value": "-", "count": None},
                {"type": "digits", "value": None, "count": 6}
            ],
            "whole_input": True
        }
    },

    {
        "id": 18,
        "lang": "he",
        "request": 'התגובה מתחילה ב"ID", אחר כך נקודתיים ואז 4 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "ID", "count": None},
                {"type": "literal", "value": ":", "count": None},
                {"type": "digits", "value": None, "count": 4}
            ],
            "whole_input": True
        }
    },

    {
        "id": 19,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"ABC", אחר כך לוכסן ואז 3 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "ABC", "count": None},
                {"type": "literal", "value": "/", "count": None},
                {"type": "digits", "value": None, "count": 3}
            ],
            "whole_input": True
        }
    },

    {
        "id": 20,
        "lang": "he",
        "request": 'הטקסט חייב להתחיל ב"שלום" אבל אסור שיופיע בו "להתראות"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "שלום", "count": None},
                {"type": "not_contains", "value": "להתראות", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 21,
        "lang": "he",
        "request": 'ההודעה יכולה להכיל כל דבר אבל אסור שתכיל "ERROR"',
        "expected": {
            "conditions": [
                {"type": "not_contains", "value": "ERROR", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 22,
        "lang": "he",
        "request": 'התגובה חייבת להתחיל ב"טופס" ואחריו רווח ו-7 ספרות בדיוק',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "טופס", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "digits", "value": None, "count": 7}
            ],
            "whole_input": True
        }
    },

    {
        "id": 23,
        "lang": "he",
        "request": 'אני מצפה לתגובה שהיא רק "OK" או "FAIL"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["OK", "FAIL"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 24,
        "lang": "he",
        "request": 'בתחילת הטקסט חייב להיות "USER"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "USER", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 25,
        "lang": "he",
        "request": 'בסוף התגובה אני רוצה לראות "SUCCESS"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "SUCCESS", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 26,
        "lang": "he",
        "request": 'התשובה היחידה שמותר לקבל היא "בוצע"',
        "expected": {
            "conditions": [
                {"type": "exact", "value": "בוצע", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 27,
        "lang": "he",
        "request": 'התשובה צריכה להיות אחת מהאפשרויות "א", "ב" או "ג"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["א", "ב", "ג"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 28,
        "lang": "he",
        "request": 'אני צריך שההודעה תכיל את הטקסט "123-ABC"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "123-ABC", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 29,
        "lang": "he",
        "request": 'ההודעה מתחילה ב"לקוח", אחריה רווח ואז "פעיל"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "לקוח", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "literal", "value": "פעיל", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 30,
        "lang": "he",
        "request": 'הטקסט צריך להתחיל ב"A" ובהמשך להכיל "B"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "A", "count": None},
                {"type": "contains_after", "value": "B", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 31,
        "lang": "he",
        "request": 'רק 6 ספרות מותרות בתגובה',
        "expected": {
            "conditions": [
                {"type": "digits", "value": None, "count": 6}
            ],
            "whole_input": True
        }
    },

    {
        "id": 32,
        "lang": "he",
        "request": 'התגובה היא בדיוק אחת מהאפשרויות "1", "2", "3" או "4"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["1", "2", "3", "4"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 33,
        "lang": "he",
        "request": 'הטקסט חייב להכיל "שלום" ואסור שיכיל "שגיאה"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "שלום", "count": None},
                {"type": "not_contains", "value": "שגיאה", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 34,
        "lang": "he",
        "request": 'ההודעה חייבת להתחיל ב"REF", אחר כך מקף ואז 8 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "REF", "count": None},
                {"type": "literal", "value": "-", "count": None},
                {"type": "digits", "value": None, "count": 8}
            ],
            "whole_input": True
        }
    },

    {
        "id": 35,
        "lang": "he",
        "request": 'התגובה חייבת להסתיים ב"מאושר"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "מאושר", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 36,
        "lang": "he",
        "request": 'חייב להופיע "ABC" איפשהו בתגובה',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "ABC", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 37,
        "lang": "he",
        "request": 'אסור שהתגובה תכיל את המילה "ביטול"',
        "expected": {
            "conditions": [
                {"type": "not_contains", "value": "ביטול", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 38,
        "lang": "he",
        "request": 'התגובה צריכה להיות בדיוק "פעיל" או "לא פעיל"',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["פעיל", "לא פעיל"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 39,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"X", אחריו נקודתיים ואז בדיוק 2 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "X", "count": None},
                {"type": "literal", "value": ":", "count": None},
                {"type": "digits", "value": None, "count": 2}
            ],
            "whole_input": True
        }
    },

    {
        "id": 40,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"HELLO", ובהמשך חייב להיות "WORLD", אבל אסור שיהיה "ERROR"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "HELLO", "count": None},
                {"type": "contains_after", "value": "WORLD", "count": None},
                {"type": "not_contains", "value": "ERROR", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 41,
        "lang": "he",
        "request": 'אני רוצה בדיוק 9 ספרות בלי שום דבר נוסף',
        "expected": {
            "conditions": [
                {"type": "digits", "value": None, "count": 9}
            ],
            "whole_input": True
        }
    },

    {
        "id": 42,
        "lang": "he",
        "request": 'התגובה חייבת להתחיל ב"מספר", אחריו רווח ואז 3 ספרות בדיוק',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "מספר", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "digits", "value": None, "count": 3}
            ],
            "whole_input": True
        }
    },

    {
        "id": 43,
        "lang": "he",
        "request": 'מותר לענות רק "ראשון", "שני" או "שלישי"',
        "expected": {
            "conditions": [
                {
                    "type": "one_of",
                    "value": ["ראשון", "שני", "שלישי"],
                    "count": None
                }
            ],
            "whole_input": True
        }
    },

    {
        "id": 44,
        "lang": "he",
        "request": 'הטקסט צריך להסתיים ב"123"',
        "expected": {
            "conditions": [
                {"type": "ends_with", "value": "123", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 45,
        "lang": "he",
        "request": 'ההודעה חייבת להתחיל ב"#"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "#", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 46,
        "lang": "he",
        "request": 'התגובה חייבת להיות בדיוק "A/B"',
        "expected": {
            "conditions": [
                {"type": "exact", "value": "A/B", "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 47,
        "lang": "he",
        "request": 'הטקסט מכיל "START" ובהמשך צריך להופיע "STOP"',
        "expected": {
            "conditions": [
                {"type": "contains", "value": "START", "count": None},
                {"type": "contains_after", "value": "STOP", "count": None}
            ],
            "whole_input": False
        }
    },

    {
        "id": 48,
        "lang": "he",
        "request": 'התגובה מתחילה ב"קוד", רווח, 4 ספרות, מקף ואז 2 ספרות',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "קוד", "count": None},
                {"type": "space", "value": None, "count": None},
                {"type": "digits", "value": None, "count": 4},
                {"type": "literal", "value": "-", "count": None},
                {"type": "digits", "value": None, "count": 2}
            ],
            "whole_input": True
        }
    },

    {
        "id": 49,
        "lang": "he",
        "request": 'התשובה חייבת להיות "פתוח" או "סגור" בלבד',
        "expected": {
            "conditions": [
                {"type": "one_of", "value": ["פתוח", "סגור"], "count": None}
            ],
            "whole_input": True
        }
    },

    {
        "id": 50,
        "lang": "he",
        "request": 'הטקסט מתחיל ב"BEGIN", בהמשך מופיע "DATA" ובסוף חייב להיות "END"',
        "expected": {
            "conditions": [
                {"type": "starts_with", "value": "BEGIN", "count": None},
                {"type": "contains_after", "value": "DATA", "count": None},
                {"type": "ends_with", "value": "END", "count": None}
            ],
            "whole_input": False
        }
    },
]

In [ ]:

print(
    "allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB | reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

allocated: 6.14 GB | reserved: 13.58 GB


In [ ]:

import time

results = []

start_time = time.time()

for test in TEST_CASES:

    print("=" * 70)
    print(f'Test {test["id"]} [{test["lang"]}]')
    print("Request:", test["request"])

    try:
        actual = understand_condition( test["request"] )
       # actual = normalize_dsl(actual)

        actual = normalize_dsl(actual,test["request"])
        expected = test["expected"]

        passed = normalize(actual) == normalize(expected)

        print("\nExpected:")
        print(
            json.dumps(
                expected,
                ensure_ascii=False,
                indent=2
            )
        )

        print("\nActual:")
        print(
            json.dumps(
                actual,
                ensure_ascii=False,
                indent=2
            )
        )

        print(
            "\nResult:",
            "✅ PASS" if passed else "❌ FAIL"
        )

        results.append({
            "id": test["id"],
            "lang": test["lang"],
            "request": test["request"],
            "expected": expected,
            "actual": actual,
            "passed": passed
        })

    except Exception as e:

        print("\n❌ ERROR:", e)

        results.append({
            "id": test["id"],
            "lang": test["lang"],
            "request": test["request"],
            "expected": test["expected"],
            "actual": None,
            "passed": False,
            "error": str(e)
        })


elapsed = time.time() - start_time

passed_count = sum(r["passed"] for r in results)
failed_count = len(results) - passed_count

print("\n" + "=" * 70)
print(f"Passed: {passed_count}/{len(results)}")
print(f"Failed: {failed_count}")
print(f"Accuracy: {passed_count / len(results) * 100:.1f}%")
print(f"Time: {elapsed:.1f} sec")

Test 1 [he]
Request: הטקסט חייב להתחיל ב"שלום"

Expected:
{
  "conditions": [
    {
      "type": "starts_with",
      "value": "שלום",
      "count": null
    }
  ],
  "whole_input": false
}

Actual:
{
  "conditions": [
    {
      "type": "starts_with",
      "value": "שלום",
      "count": null
    }
  ],
  "whole_input": false
}

Result: ✅ PASS
Test 2 [he]
Request: ההודעה צריכה להתחיל במילה "הזמנה"

Expected:
{
  "conditions": [
    {
      "type": "starts_with",
      "value": "הזמנה",
      "count": null
    }
  ],
  "whole_input": false
}

Actual:
{
  "conditions": [
    {
      "type": "starts_with",
      "value": "הזמנה",
      "count": null
    }
  ],
  "whole_input": false
}

Result: ✅ PASS
Test 3 [he]
Request: הטקסט צריך להסתיים ב"תודה"

Expected:
{
  "conditions": [
    {
      "type": "ends_with",
      "value": "תודה",
      "count": null
    }
  ],
  "whole_input": false
}

Actual:
{
  "conditions": [
    {
      "type": "ends_with",
      "value": "תודה",
      "count

In [ ]:

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(
    "allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB | reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

allocated: 0.0 GB | reserved: 0.0 GB


In [19]:

result = understand_condition(
    'התגובה מתחילה ב"ID", אחר כך נקודתיים ואז 4 ספרות'
)

print(json.dumps(
    result,
    ensure_ascii=False,
    indent=2
))

{
  "conditions": [
    {
      "type": "starts_with",
      "value": "ID",
      "count": null
    },
    {
      "type": "literal",
      "value": ".",
      "count": null
    },
    {
      "type": "digits",
      "value": null,
      "count": 4
    }
  ],
  "whole_input": true
}
